In [ ]:
import os
import joblib
import pandas as pd
import mysql.connector
import boto3
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "mysql_db"),
    user=os.getenv("MYSQL_USER", "airflow"),
    password=os.getenv("MYSQL_PASSWORD", "airflow"),
    database=os.getenv("MYSQL_DATABASE", "covertype_data")
)
df = pd.read_sql("SELECT * FROM covertype_cleaned WHERE dataset = 'train'", conn)
conn.close()

In [ ]:
drop_cols = ["id", "dataset", "cover_type"]
feature_cols = [c for c in df.columns if c not in drop_cols]
X_train = df[feature_cols]
y_train = df["cover_type"]

In [ ]:
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "mysql_db"),
    user=os.getenv("MYSQL_USER", "airflow"),
    password=os.getenv("MYSQL_PASSWORD", "airflow"),
    database=os.getenv("MYSQL_DATABASE", "covertype_data")
)
df_test = pd.read_sql("SELECT * FROM covertype_cleaned WHERE dataset = 'test'", conn)
conn.close()
X_test = df_test[feature_cols]
y_test = df_test["cover_type"]

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
bucket = os.getenv("MINIO_BUCKET", "covertype-project")
model_name = os.getenv("MODEL_NAME", "random_forest")
tmp_path = f"/tmp/{model_name}.joblib"
joblib.dump(model, tmp_path)
client = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
)
client.upload_file(tmp_path, bucket, f"models/{model_name}.joblib")